In [1]:
import pandas as pd
import numpy as np

# --- CARGA DE DATOS DESDE RUTA ESPECÍFICA ---
FILE_PATH_PARQUET = '/home/quant/data/processed/nq_1m_continuous.parquet'

try:
    print(f"📂 Cargando datos desde: {FILE_PATH_PARQUET}...")
    df_raw = pd.read_parquet(FILE_PATH_PARQUET)
    

except FileNotFoundError:
    print(f"🛑 Error: No se encontró el archivo en {FILE_PATH_PARQUET}")
except Exception as e:
    print(f"🛑 Error durante la carga o análisis: {e}")
df_raw.head()

📂 Cargando datos desde: /home/quant/data/processed/nq_1m_continuous.parquet...


,Open,High,Low,Close,Volume,Ticker
timestamp,,,,,,
2010-06-06 18:00:00,1831.75,1831.75,1828.75,1829.00,117,NQM0
2010-06-06 18:01:00,1828.50,1831.00,1828.50,1830.50,99,NQM0
2010-06-06 18:02:00,1830.50,1832.25,1830.50,1831.25,32,NQM0
2010-06-06 18:03:00,1830.50,1831.50,1830.50,1830.50,12,NQM0
2010-06-06 18:04:00,1830.50,1830.50,1830.50,1830.50,1,NQM0


In [6]:
import pandas as pd
import numpy as np
from tqdm import auto as tqdm # Barra de progreso para procesamiento intensivo
from IPython.display import display, Markdown

def compute_vwap(df):
    """Calcula VWAP diario reiniciado a las 09:30"""
    # Precio Típico para el cálculo de VWAP utilizando nombres exactos del DF
    p = (df['High'] + df['Low'] + df['Close']) / 3
    v = df['Volume']
    
    # Agrupación por día para reinicio de VWAP
    df['vwap'] = (p * v).groupby(df.index.date).cumsum() / v.groupby(df.index.date).cumsum()
    return df

def analyze_orbit_regime(df):
    """
    Transforma ideas de trading en validación estadística.
    Objetivo: Validar el umbral de 7 cruces para reversión PM.
    """
    print("⏳ Iniciando pre-procesamiento y alineación RTH...")
    
    # Verificación de integridad del índice
    if not isinstance(df.index, pd.DatetimeIndex):
        print("🔧 Convirtiendo índice a DatetimeIndex...")
        df.index = pd.to_datetime(df.index)
    
    # REQ-01: Filtrar RTH (09:30 a 16:00 EST)
    df_rth = df.between_time('09:30', '16:00').copy()
    
    if df_rth.empty:
        print("🛑 Error: El filtrado RTH resultó en un DataFrame vacío. Verifica el horario de tus datos.")
        return None

    # REQ-02: Reiniciar VWAP diariamente
    df_rth = compute_vwap(df_rth)
    
    # Feature Engineering (Nombres de columnas respetados)
    df_rth['dist_vwap'] = df_rth['Close'] - df_rth['vwap']
    
    # Z-Score Dinámico (Rolling 20 periodos sobre la distancia al VWAP)
    df_rth['rolling_std'] = df_rth['dist_vwap'].groupby(df_rth.index.date).transform(lambda x: x.rolling(20).std())
    df_rth['z_score'] = df_rth['dist_vwap'] / df_rth['rolling_std']
    
    # Contador de Cruces de VWAP
    df_rth['cross'] = (np.sign(df_rth['dist_vwap']) != np.sign(df_rth['dist_vwap'].shift(1))).astype(int)
    # Evitar el primer cruce falso por el shift inicial
    df_rth.iloc[0, df_rth.columns.get_loc('cross')] = 0
    df_rth['daily_crosses'] = df_rth['cross'].groupby(df_rth.index.date).cumsum()

    # Segmentación Temporal: Mañana (AM) vs Tarde (PM)
    df_am = df_rth.between_time('09:30', '12:00')
    df_pm = df_rth.between_time('12:01', '16:00')

    daily_stats = []
    dates = df_am.index.date.unique()
    
    print(f"🔬 Analizando {len(dates)} jornadas operativas...")
    
    # Procesamiento con Barra de Avance
    for date in tqdm.tqdm(dates, desc="Procesando Buckets"):
        group_am = df_am[df_am.index.date == date]
        if group_am.empty: continue
        
        # Cruces acumulados detectados hasta el mediodía (12:00)
        crosses_am = group_am['daily_crosses'].iloc[-1]
        
        group_pm = df_pm[df_pm.index.date == date]
        if group_pm.empty: continue
            
        # INVESTIGACIÓN: ¿Hubo oportunidad de reversión tras sobreextensión?
        # Z > 2.0 en la tarde
        opportunity_pm = (group_pm['z_score'].abs() > 2.0).any()
        
        success_pm = False
        if opportunity_pm:
            # Encontrar el primer momento de sobreextensión
            idx_extrema = group_pm['z_score'].abs() > 2.0
            first_extrema_time = group_pm[idx_extrema].index[0]
            # Verificamos retorno al VWAP (Z < 0.2) DESPUÉS de ese momento
            post_extrema = group_pm.loc[first_extrema_time:]
            success_pm = (post_extrema['z_score'].abs() < 0.2).any()
        
        daily_stats.append({
            'date': date,
            'crosses_am': crosses_am,
            'opportunity_pm': opportunity_pm,
            'success_pm': success_pm
        })

    # Consolidación de Resultados
    stats_df = pd.DataFrame(daily_stats)
    
    if stats_df.empty:
        print("🛑 No se pudieron generar estadísticas.")
        return None

    # Clasificación por Buckets de Probabilidad (REQ-03)
    bins = [-1, 3, 6, 9, 15, 1000]
    labels = ['0-3 (Trend)', '4-6 (Transición)', '7-9 (Órbita-🎯)', '10-15 (Range)', '15+ (Noise)']
    stats_df['cross_bucket'] = pd.cut(stats_df['crosses_am'], bins=bins, labels=labels)
    
    # Generación de Tabla de Contingencia
    report = stats_df.groupby('cross_bucket', observed=False).agg(
        Dias_Totales=('date', 'count'),
        Oportunidades_Z2_PM=('opportunity_pm', 'sum'),
        Reversiones_Exitosas_PM=('success_pm', 'sum')
    )
    
    # Probabilidad Condicional: P(Reversión | Extensión Z2)
    report['Prob_Exito_Reversion'] = (report['Reversiones_Exitosas_PM'] / report['Oportunidades_Z2_PM'].replace(0, np.nan)).round(4)
    
    display(Markdown("### 📊 Resultado de la Investigación: Buckets de Probabilidad NQ Orbit"))
    display(report)
    
    return report

# --- BLOQUE DE EJECUCIÓN ---
if 'df_raw' in globals():
    resultado_investigacion = analyze_orbit_regime(df_raw)
else:
    print("❌ Error: No se detectó la variable 'df_raw' en memoria.")

⏳ Iniciando pre-procesamiento y alineación RTH...


AttributeError: 'numpy.ndarray' object has no attribute 'unique'

In [7]:
import pandas as pd
import numpy as np
from tqdm import auto as tqdm # Barra de progreso para procesamiento intensivo
from IPython.display import display, Markdown

def compute_vwap(df):
    """Calcula VWAP diario reiniciado a las 09:30"""
    # Precio Típico para el cálculo de VWAP utilizando nombres exactos del DF
    p = (df['High'] + df['Low'] + df['Close']) / 3
    v = df['Volume']
    
    # Agrupación por día para reinicio de VWAP
    df['vwap'] = (p * v).groupby(df.index.date).cumsum() / v.groupby(df.index.date).cumsum()
    return df

def analyze_orbit_regime(df):
    """
    Transforma ideas de trading en validación estadística.
    Objetivo: Validar el umbral de 7 cruces para reversión PM.
    """
    print("⏳ Iniciando pre-procesamiento y alineación RTH...")
    
    # Verificación de integridad del índice
    if not isinstance(df.index, pd.DatetimeIndex):
        print("🔧 Convirtiendo índice a DatetimeIndex...")
        df.index = pd.to_datetime(df.index)
    
    # REQ-01: Filtrar RTH (09:30 a 16:00 EST)
    df_rth = df.between_time('09:30', '16:00').copy()
    
    if df_rth.empty:
        print("🛑 Error: El filtrado RTH resultó en un DataFrame vacío. Verifica el horario de tus datos.")
        return None

    # REQ-02: Reiniciar VWAP diariamente
    df_rth = compute_vwap(df_rth)
    
    # Feature Engineering (Nombres de columnas respetados)
    df_rth['dist_vwap'] = df_rth['Close'] - df_rth['vwap']
    
    # Z-Score Dinámico (Rolling 20 periodos sobre la distancia al VWAP)
    df_rth['rolling_std'] = df_rth['dist_vwap'].groupby(df_rth.index.date).transform(lambda x: x.rolling(20).std())
    df_rth['z_score'] = df_rth['dist_vwap'] / df_rth['rolling_std']
    
    # Contador de Cruces de VWAP
    df_rth['cross'] = (np.sign(df_rth['dist_vwap']) != np.sign(df_rth['dist_vwap'].shift(1))).astype(int)
    # Evitar el primer cruce falso por el shift inicial
    df_rth.iloc[0, df_rth.columns.get_loc('cross')] = 0
    df_rth['daily_crosses'] = df_rth['cross'].groupby(df_rth.index.date).cumsum()

    # Segmentación Temporal: Mañana (AM) vs Tarde (PM)
    df_am = df_rth.between_time('09:30', '12:00')
    df_pm = df_rth.between_time('12:01', '16:00')

    daily_stats = []
    # CORRECCIÓN: Asegurar el uso de unique() sobre un tipo compatible
    dates = pd.Series(df_am.index.date).unique()
    
    print(f"🔬 Analizando {len(dates)} jornadas operativas...")
    
    # Procesamiento con Barra de Avance
    for date in tqdm.tqdm(dates, desc="Procesando Buckets"):
        group_am = df_am[df_am.index.date == date]
        if group_am.empty: continue
        
        # Cruces acumulados detectados hasta el mediodía (12:00)
        crosses_am = group_am['daily_crosses'].iloc[-1]
        
        group_pm = df_pm[df_pm.index.date == date]
        if group_pm.empty: continue
            
        # INVESTIGACIÓN: ¿Hubo oportunidad de reversión tras sobreextensión?
        # Z > 2.0 en la tarde
        opportunity_pm = (group_pm['z_score'].abs() > 2.0).any()
        
        success_pm = False
        if opportunity_pm:
            # Encontrar el primer momento de sobreextensión
            idx_extrema = group_pm['z_score'].abs() > 2.0
            first_extrema_time = group_pm[idx_extrema].index[0]
            # Verificamos retorno al VWAP (Z < 0.2) DESPUÉS de ese momento
            post_extrema = group_pm.loc[first_extrema_time:]
            success_pm = (post_extrema['z_score'].abs() < 0.2).any()
        
        daily_stats.append({
            'date': date,
            'crosses_am': crosses_am,
            'opportunity_pm': opportunity_pm,
            'success_pm': success_pm
        })

    # Consolidación de Resultados
    stats_df = pd.DataFrame(daily_stats)
    
    if stats_df.empty:
        print("🛑 No se pudieron generar estadísticas.")
        return None

    # Clasificación por Buckets de Probabilidad (REQ-03)
    bins = [-1, 3, 6, 9, 15, 1000]
    labels = ['0-3 (Trend)', '4-6 (Transición)', '7-9 (Órbita-🎯)', '10-15 (Range)', '15+ (Noise)']
    stats_df['cross_bucket'] = pd.cut(stats_df['crosses_am'], bins=bins, labels=labels)
    
    # Generación de Tabla de Contingencia
    report = stats_df.groupby('cross_bucket', observed=False).agg(
        Dias_Totales=('date', 'count'),
        Oportunidades_Z2_PM=('opportunity_pm', 'sum'),
        Reversiones_Exitosas_PM=('success_pm', 'sum')
    )
    
    # Probabilidad Condicional: P(Reversión | Extensión Z2)
    report['Prob_Exito_Reversion'] = (report['Reversiones_Exitosas_PM'] / report['Oportunidades_Z2_PM'].replace(0, np.nan)).round(4)
    
    display(Markdown("### 📊 Resultado de la Investigación: Buckets de Probabilidad NQ Orbit"))
    display(report)
    
    return report

# --- BLOQUE DE EJECUCIÓN ---
if 'df_raw' in globals():
    resultado_investigacion = analyze_orbit_regime(df_raw)
else:
    print("❌ Error: No se detectó la variable 'df_raw' en memoria.")

⏳ Iniciando pre-procesamiento y alineación RTH...
🔬 Analizando 3988 jornadas operativas...


Procesando Buckets:   0%|          | 0/3988 [00:00<?, ?it/s]

### 📊 Resultado de la Investigación: Buckets de Probabilidad NQ Orbit

,Dias_Totales,Oportunidades_Z2_PM,Reversiones_Exitosas_PM,Prob_Exito_Reversion
cross_bucket,,,,
0-3 (Trend),460,460,248,0.5391
4-6 (Transición),719,719,419,0.5828
7-9 (Órbita-🎯),796,796,504,0.6332
10-15 (Range),1272,1272,863,0.6785
15+ (Noise),725,725,502,0.6924
